# Bit Flip Mutation : Best For F18

In [ ]:
import numpy as np
# you need to install this package `ioh`. Please see documentations here: 
# https://iohprofiler.github.io/IOHexp/ and
# https://pypi.org/project/ioh/
from ioh import get_problem, logger, ProblemClass
import itertools

import sys
# budget = 5000
dimension = 50

# seletype = 'proportional'
# cross_type = 'uniform'

# crossover_probability=0.2
# mutation_rate = 0.02
# pop_size = 10
# seed = 42 #456 #123

# tournament_k = 5
# npoints = 3 #5 #3


# np.random.seed(seed)


# # To make your results reproducible (not required by the assignment), you could set the random seed by
# np.random.seed(42)
# Uniform Crossover
def single_point_crossover(p1,p2,x):
    p1_new=np.append(p1[:x],p2[x:])
    p2_new=np.append(p2[:x],p1[x:])
    return p1_new,p2_new


def uniform_crossover(p1, p2, crossover_probability):
    if cross_type == 'uniform':
        if np.random.uniform(0, 1) < crossover_probability:
            for i in range(len(p1)):
                if np.random.uniform(0, 1) < 0.5:
                    t = p1[i]
                    p1[i] = p2[i]
                    p2[i] = t

def npoints_crossover(p1, p2, crossover_probability, npoints):
    if np.random.uniform(0, 1) < crossover_probability:
        cross_points = list(np.random.choice(range(1, 50), npoints, replace=False))
        for i in cross_points:
            p1, p2 = single_point_crossover(p1.copy(), p2.copy(), i) 
    return p1, p2
            

# Standard bit mutation using mutation rate p
def mutation(p, mutation_rate):
    for i in range(len(p)) :
        if np.random.uniform(0,1) < mutation_rate:
            p[i] = 1 - p[i]
  

def proportional_selection(parent, parent_f):
    # Plusing 0.001 to avoid dividing 0
    f_min = min(parent_f)
    f_sum = sum(parent_f) - (f_min - 0.001) * len(parent_f)
    
    rw = [(parent_f[0] - f_min + 0.001)/f_sum]
    for i in range(1,len(parent_f)):
        rw.append(rw[i-1] + (parent_f[i] - f_min + 0.001) / f_sum)
    
    select_parent = []
    for i in range(len(parent)) :
        r = np.random.uniform(0,1)
        index = 0
        # print(rw,r)
        while(r > rw[index]) :
            index = index + 1
        select_parent.append(parent[index].copy())
    return select_parent


def tournamentk_selection(parent, parent_f, tournament_k): 
    select_parent = []
    tournament_k = min(tournament_k, len(parent_f))
    pre_select = np.random.choice(len(parent_f), tournament_k, replace=False)
    index = 0 
    for i in range(len(parent)) :
        pre_select = np.random.choice(len(parent_f),tournament_k,replace = False)
        max_f = sys.float_info.min
        for p in pre_select:
            if parent_f[p] > max_f:
                index = p
                max_f = parent_f[p]
        select_parent.append(parent[index].copy())
    return select_parent



def studentnumber1_studentnumber2_GA(problem,  pop_size, budget = None):
    # initial_pop = ... make sure you randomly create the first population
    # `problem.state.evaluations` counts the number of function evaluation automatically,
    # which is incremented by 1 whenever you call `problem(x)`.
    # You could also maintain a counter of function evaluations if you prefer.
    if problem.meta_data.problem_id == 18:
        optimum=8
    if problem.meta_data.problem_id == 19:
        optimum=50
    if budget is None:
        budget = 5000
    
    f_opt = sys.float_info.min
    x_opt = None
    
    parent = []
    parent_f = []
    for i in range(pop_size):

        # Initialization
        parent.append(np.random.randint(2, size = problem.meta_data.n_variables))
        parent_f.append(problem(parent[i]))
        # budget = budget - 1


    while problem.state.evaluations < budget:
        # please implement the mutation, crossover, selection here
        # .....
        # this is how you evaluate one solution `x`
        # f = problem(x)
        # Evaluate fitness for each individual in the population
    # no return value needed 
        if seletype == 'proportional':
            offspring = proportional_selection(parent,parent_f)
        elif seletype == 'tournamentk':
            offspring = tournamentk_selection(parent,parent_f,tournament_k)

        if cross_type=='npoints':
            for i in range(0,pop_size - (pop_size%2),2) :
                offspring[i], offspring[i+1]  =  npoints_crossover(offspring[i], offspring[i+1], crossover_probability, npoints) 
        elif cross_type=='uniform':
            for i in range(0,pop_size - (pop_size%2),2) :
                uniform_crossover(offspring[i], offspring[i+1], crossover_probability) 


        for i in range(pop_size):
            mutation(offspring[i], mutation_rate)

        parent = offspring.copy()
        for i in range(pop_size) : 
            parent_f[i] = problem(parent[i])
            # budget = budget - 1
            if parent_f[i] > f_opt:
                    f_opt = parent_f[i]
                    x_opt = parent[i].copy()
            if f_opt >= optimum or problem.state.evaluations >= budget:
                break
        
    print(f_opt) #x_opt
    return f_opt

def create_problem(fid: int):
    # Declaration of problems to be tested.
    problem = get_problem(fid, dimension=dimension, instance=1, problem_class=ProblemClass.PBO)

    # Create default logger compatible with IOHanalyzer
    # `root` indicates where the output files are stored.
    # `folder_name` is the name of the folder containing all output. You should compress the folder 'run' and upload it to IOHanalyzer.
    l = logger.Analyzer(
        root="data",  # the working directory in which a folder named `folder_name` (the next argument) will be created to store data
        folder_name=f"run_f{fid}",  # the folder name to which the raw performance data will be stored
        algorithm_name=f"ga_f{fid}",  # name of your algorithm
        algorithm_info=f"Practical assignment of the EA course f{fid}",
    )
    # attach the logger to the problem
    problem.attach_logger(l)
    return problem, l



if __name__ == "__main__":
    # Define the hyperparameter grid

    seletype = 'tournamentk'
    cross_type = 'npoints' 

    # crossover_probability=0.2
    # mutation_rate = 0.02
    # pop_size = 10
    # seed = 42 #456 #123

    # tournament_k = 5
    # npoints = 3 #5 #3

    pop_size_values = [10, 30, 45]  # Add more values as needed
    crossover_probability_values = [0.2, 0.5, 0.8]  # Add more values as needed
    mutation_rate_values = [0.02, 0.05]  # Add more values as needed
    seeds = [42]
    tournamentk_values = [5, 20, 35]
    npoints_values = [5, 10]

    # Perform grid search
    results = []
    for pop_size, crossover_prob, mutation_rate, seed, tournament_k, npoints in itertools.product(pop_size_values, crossover_probability_values, mutation_rate_values, seeds, tournamentk_values,npoints_values):

        crossover_probability=crossover_prob
        mutation_rate=mutation_rate
        tournament_k= tournament_k
        npoints = npoints
        np.random.seed(seed)

        F18, _logger = create_problem(18)
        #optimum=15
        s_f_opt=0
        for run in range(20): 
            print(f'F18, Run:{run}')
            f_opt = studentnumber1_studentnumber2_GA(F18, pop_size)
            s_f_opt=s_f_opt+f_opt
            F18.reset() # it is necessary to reset the problem after each independent run
        _logger.close() # after all runs, it is necessary to close the logger to make sure all data are written to the folder)

        np.random.seed(seed)

        F19, _logger = create_problem(19)
        #optimum=50
        s_f_opt2=0
        for run in range(20): 
            print(f'F19, Run:{run}')
            f_opt2=studentnumber1_studentnumber2_GA(F19, pop_size)
            s_f_opt2=s_f_opt2+f_opt2
            F19.reset()

        print(f"Running GA with npoints, tournamentk, pop_size={pop_size}, crossover_prob={crossover_prob}, mutation_rate={mutation_rate}, seed={seed}, tournamentk={tournament_k}, npoints={npoints} ")
        print('F18 fitness average:',s_f_opt/20)
        print('F19 fitness average:',s_f_opt2/20)
        f_opt_f18 = s_f_opt/20
        f_opt_f19 = s_f_opt2/20
        results.append({
            'npoints:': npoints,
            'tournament_k':tournament_k,
            'pop_size': pop_size,
            'crossover_prob': crossover_prob,
            'mutation_rate': mutation_rate,
            'seed' : seed,
            'f_opt_f18': f_opt_f18,
            'f_opt_f19': f_opt_f19
        })
        _logger.close()

    print("\nTop 5 Combinations based on F18 Fitness:")
    results.sort(key=lambda x: x['f_opt_f18'], reverse=True)
    for i in range(min(5, len(results))):
        print(f"Combination {i + 1}: {results[i]}")

    print("\nTop 5 Combinations based on F19 Fitness:")
    results.sort(key=lambda x: x['f_opt_f19'], reverse=True)
    for i in range(min(5, len(results))):
        print(f"Combination {i + 1}: {results[i]}")


F18, Run:0
4.448398576512456
F18, Run:1
4.325259515570934
F18, Run:2
3.9936102236421727
F18, Run:3
4.325259515570934
F18, Run:4
4.2087542087542085
F18, Run:5
3.8940809968847354
F18, Run:6
4.716981132075472
F18, Run:7
3.7993920972644375
F18, Run:8
3.9936102236421727
F18, Run:9
5.020080321285141
F18, Run:10
3.5410764872521248
F18, Run:11
4.448398576512456
F18, Run:12
4.2087542087542085
F18, Run:13
3.9936102236421727
F18, Run:14
4.098360655737705
F18, Run:15
5.186721991701245
F18, Run:16
3.9936102236421727
F18, Run:17
4.5787545787545785
F18, Run:18
3.8940809968847354
F18, Run:19
3.7091988130563798
F19, Run:0
48.0
F19, Run:1
46.0
F19, Run:2
46.0
F19, Run:3
48.0
F19, Run:4
48.0
F19, Run:5
46.0
F19, Run:6
50.0
F19, Run:7
46.0
F19, Run:8
48.0
F19, Run:9
48.0
F19, Run:10
48.0
F19, Run:11
46.0
F19, Run:12
48.0
F19, Run:13
46.0
F19, Run:14
48.0
F19, Run:15
48.0
F19, Run:16
48.0
F19, Run:17
48.0
F19, Run:18
44.0
F19, Run:19
48.0
Running GA with npoints, tournamentk, pop_size=10, crossover_prob=0.

# Binomial mutation : Best for F19

In [2]:
import numpy as np
# you need to install this package `ioh`. Please see documentations here: 
# https://iohprofiler.github.io/IOHexp/ and
# https://pypi.org/project/ioh/
from ioh import get_problem, logger, ProblemClass
import itertools
import sys
# budget = 5000
dimension = 50

# seletype = 'proportional'
# cross_type = 'uniform'

# crossover_probability=0.2
# mutation_rate = 0.02
# pop_size = 10
# seed = 42 #456 #123

# tournament_k = 5
# npoints = 3 #5 #3


# np.random.seed(seed)


# # To make your results reproducible (not required by the assignment), you could set the random seed by
# np.random.seed(42)
# Uniform Crossover
def single_point_crossover(p1,p2,x):
    p1_new=np.append(p1[:x],p2[x:])
    p2_new=np.append(p2[:x],p1[x:])
    return p1_new,p2_new


def uniform_crossover(p1, p2, crossover_probability):
    if cross_type == 'uniform':
        if np.random.uniform(0, 1) < crossover_probability:
            for i in range(len(p1)):
                if np.random.uniform(0, 1) < 0.5:
                    t = p1[i]
                    p1[i] = p2[i]
                    p2[i] = t

def npoints_crossover(p1, p2, crossover_probability, npoints):
    if np.random.uniform(0, 1) < crossover_probability:
        cross_points = list(np.random.choice(range(1, 50), npoints, replace=False))
        for i in cross_points:
            p1, p2 = single_point_crossover(p1.copy(), p2.copy(), i) 
    return p1, p2
            

# # Standard bit mutation using mutation rate p
# def mutation(p, mutation_rate):
#     for i in range(len(p)) :
#         if np.random.uniform(0,1) < mutation_rate:
#             p[i] = 1 - p[i]

def binomial_mutation(individual, mutation_probability, mutation_strength):
    mutated_individual = individual.copy()

    for i in range(len(mutated_individual)):
        if np.random.rand() < mutation_probability:
            # Add a random value from a binomial distribution
            mutation_value = np.random.binomial(1, mutation_strength)
            mutated_individual[i] = (mutated_individual[i] + mutation_value) % 2

    return mutated_individual
  

def proportional_selection(parent, parent_f):
    # Plusing 0.001 to avoid dividing 0
    f_min = min(parent_f)
    f_sum = sum(parent_f) - (f_min - 0.001) * len(parent_f)
    
    rw = [(parent_f[0] - f_min + 0.001)/f_sum]
    for i in range(1,len(parent_f)):
        rw.append(rw[i-1] + (parent_f[i] - f_min + 0.001) / f_sum)
    
    select_parent = []
    for i in range(len(parent)) :
        r = np.random.uniform(0,1)
        index = 0
        # print(rw,r)
        while(r > rw[index]) :
            index = index + 1
        select_parent.append(parent[index].copy())
    return select_parent


def tournamentk_selection(parent, parent_f, tournament_k): 
    select_parent = []
    tournament_k = min(tournament_k, len(parent_f))
    pre_select = np.random.choice(len(parent_f), tournament_k, replace=False)
    index = 0 
    for i in range(len(parent)) :
        pre_select = np.random.choice(len(parent_f),tournament_k,replace = False)
        max_f = sys.float_info.min
        for p in pre_select:
            if parent_f[p] > max_f:
                index = p
                max_f = parent_f[p]
        select_parent.append(parent[index].copy())
    return select_parent



def studentnumber1_studentnumber2_GA(problem,  pop_size, budget = None):
    # initial_pop = ... make sure you randomly create the first population
    # `problem.state.evaluations` counts the number of function evaluation automatically,
    # which is incremented by 1 whenever you call `problem(x)`.
    # You could also maintain a counter of function evaluations if you prefer.
    if problem.meta_data.problem_id == 18:
        optimum=8
    if problem.meta_data.problem_id == 19:
        optimum=50
    if budget is None:
        budget = 5000
    
    f_opt = sys.float_info.min
    x_opt = None
    
    parent = []
    parent_f = []
    for i in range(pop_size):

        # Initialization
        parent.append(np.random.randint(2, size = problem.meta_data.n_variables))
        parent_f.append(problem(parent[i]))
        # budget = budget - 1


    while problem.state.evaluations < budget:
        # please implement the mutation, crossover, selection here
        # .....
        # this is how you evaluate one solution `x`
        # f = problem(x)
        # Evaluate fitness for each individual in the population
    # no return value needed 
        if seletype == 'proportional':
            offspring = proportional_selection(parent,parent_f)
        elif seletype == 'tournamentk':
            offspring = tournamentk_selection(parent,parent_f,tournament_k)

        if cross_type=='npoints':
            for i in range(0,pop_size - (pop_size%2),2) :
                offspring[i], offspring[i+1]  =  npoints_crossover(offspring[i], offspring[i+1], crossover_probability, npoints) 
        elif cross_type=='uniform':
            for i in range(0,pop_size - (pop_size%2),2) :
                uniform_crossover(offspring[i], offspring[i+1], crossover_probability) 


        for i in range(pop_size):
            offspring[i] =binomial_mutation(offspring[i], mutation_rate, mutation_strength)

        parent = offspring.copy()
        for i in range(pop_size) : 
            parent_f[i] = problem(parent[i])
            # budget = budget - 1
            if parent_f[i] > f_opt:
                    f_opt = parent_f[i]
                    x_opt = parent[i].copy()
            if f_opt >= optimum or problem.state.evaluations >= budget:
                break
        
    print(f_opt) #x_opt
    return f_opt

def create_problem(fid: int):
    # Declaration of problems to be tested.
    problem = get_problem(fid, dimension=dimension, instance=1, problem_class=ProblemClass.PBO)

    # Create default logger compatible with IOHanalyzer
    # `root` indicates where the output files are stored.
    # `folder_name` is the name of the folder containing all output. You should compress the folder 'run' and upload it to IOHanalyzer.
    l = logger.Analyzer(
        root="data",  # the working directory in which a folder named `folder_name` (the next argument) will be created to store data
        folder_name=f"run_f{fid}",  # the folder name to which the raw performance data will be stored
        algorithm_name=f"ga_f{fid}",  # name of your algorithm
        algorithm_info=f"Practical assignment of the EA course f{fid}",
    )
    # attach the logger to the problem
    problem.attach_logger(l)
    return problem, l



if __name__ == "__main__":
    # Define the hyperparameter grid

    seletype = 'tournamentk'
    cross_type = 'npoints' 

    # crossover_probability=0.2
    # mutation_rate = 0.02
    # pop_size = 10
    # seed = 42 #456 #123

    # tournament_k = 5
    # npoints = 3 #5 #3

    pop_size_values = [10, 30]  # Add more values as needed
    crossover_probability_values = [0.2, 0.5, 0.8]  # Add more values as needed
    mutation_rate_values = [0.02, 0.05]  # Add more values as needed
    seeds = [42]
    tournamentk_values = [5, 10, 20]
    npoints_values = [5, 10, 20]
    mutation_strengths = [ 0.5, 1.0]

    # Perform grid search
    results = []
    for pop_size, crossover_prob, mutation_rate, seed, tournament_k, npoints, mutation_strength in itertools.product(pop_size_values, crossover_probability_values, mutation_rate_values, seeds, tournamentk_values,npoints_values,mutation_strengths):

        crossover_probability=crossover_prob
        mutation_rate=mutation_rate
        tournament_k= tournament_k
        npoints = npoints
        mutation_strength=mutation_strength
        np.random.seed(seed)

        F18, _logger = create_problem(18)
        #optimum=15
        s_f_opt=0
        for run in range(20): 
            print(f'F18, Run:{run}')
            f_opt = studentnumber1_studentnumber2_GA(F18, pop_size)
            s_f_opt=s_f_opt+f_opt
            F18.reset() # it is necessary to reset the problem after each independent run
        _logger.close() # after all runs, it is necessary to close the logger to make sure all data are written to the folder


        np.random.seed(seed)
        
        F19, _logger = create_problem(19)
        #optimum=50
        s_f_opt2=0
        for run in range(20): 
            print(f'F19, Run:{run}')
            f_opt2=studentnumber1_studentnumber2_GA(F19, pop_size)
            s_f_opt2=s_f_opt2+f_opt2
            F19.reset()

        print(f"Running GA with npoints, tournamentk, pop_size={pop_size}, crossover_prob={crossover_prob}, mutation_rate={mutation_rate}, seed={seed}, tournamentk={tournament_k}, npoints={npoints},  mutation_strength={ mutation_strength}")
        print('F18 fitness average:',s_f_opt/20)
        print('F19 fitness average:',s_f_opt2/20)
        f_opt_f18 = s_f_opt/20
        f_opt_f19 = s_f_opt2/20
        results.append({
            'npoints:': npoints,
            'tournament_k':tournament_k,
            'pop_size': pop_size,
            'crossover_prob': crossover_prob,
            'mutation_rate': mutation_rate,
            'seed' : seed,
            'mutation_strength' :  mutation_strength,
            'f_opt_f18': f_opt_f18,
            'f_opt_f19': f_opt_f19
        })
        _logger.close()

    print("\nTop 5 Combinations based on F18 Fitness:")
    results.sort(key=lambda x: x['f_opt_f18'], reverse=True)
    for i in range(min(5, len(results))):
        print(f"Combination {i + 1}: {results[i]}")

    print("\nTop 5 Combinations based on F19 Fitness:")
    results.sort(key=lambda x: x['f_opt_f19'], reverse=True)
    for i in range(min(5, len(results))):
        print(f"Combination {i + 1}: {results[i]}")


F18, Run:0
3.6231884057971016
F18, Run:1
3.3875338753387534
F18, Run:2
3.8940809968847354
F18, Run:3
3.8940809968847354
F18, Run:4
3.117206982543641
F18, Run:5
3.7993920972644375
F18, Run:6
3.8940809968847354
F18, Run:7
3.315649867374005
F18, Run:8
4.098360655737705
F18, Run:9
4.2087542087542085
F18, Run:10
3.8940809968847354
F18, Run:11
3.1806615776081424
F18, Run:12
4.098360655737705
F18, Run:13
3.4626038781163433
F18, Run:14
4.098360655737705
F18, Run:15
3.4626038781163433
F18, Run:16
3.2467532467532467
F18, Run:17
3.7993920972644375
F18, Run:18
3.9936102236421727
F18, Run:19
3.315649867374005
F19, Run:0
46.0
F19, Run:1
44.0
F19, Run:2
48.0
F19, Run:3
46.0
F19, Run:4
48.0
F19, Run:5
46.0
F19, Run:6
48.0
F19, Run:7
46.0
F19, Run:8
46.0
F19, Run:9
46.0
F19, Run:10
44.0
F19, Run:11
46.0
F19, Run:12
46.0
F19, Run:13
46.0
F19, Run:14
46.0
F19, Run:15
48.0
F19, Run:16
48.0
F19, Run:17
48.0
F19, Run:18
46.0
F19, Run:19
48.0
Running GA with npoints, tournamentk, pop_size=10, crossover_prob=